In [1]:
import torch

In [2]:
BASE_CONFIG = {
    "vocab_size": 50257,    # Vocabulary size
    "context_length": 1024, # Context length
    "dropout_prob": 0.0,       # Dropout rate
    "qkv_bias": True        # Query-key-value bias
}

model_configs = {
    "gpt2-small (124M)": {"emb_dim": 768, "n_layers": 12, "n_heads": 12, "hidden_dim": 3072},
    "gpt2-medium (355M)": {"emb_dim": 1024, "n_layers": 24, "n_heads": 16, "hidden_dim": 4096},
    "gpt2-large (774M)": {"emb_dim": 1280, "n_layers": 36, "n_heads": 20, "hidden_dim": 5120},
    "gpt2-xl (1558M)": {"emb_dim": 1600, "n_layers": 48, "n_heads": 25, "hidden_dim": 6400},
}


CHOOSE_MODEL = "gpt2-small (124M)"
BASE_CONFIG.update(model_configs[CHOOSE_MODEL])

In [5]:
from architecture.gpt_model import GPTModel

filename = "./ckpt/gpt2-small-124M.pth"

gpt_model = GPTModel(BASE_CONFIG)
checkpoint = torch.load(filename, map_location="cpu", weights_only=True)

# # Translate parameter names from the checkpoint's older model implementation.
compatible_checkpoint = {}
for key, value in checkpoint.items():
    if key.endswith(".att.mask"):
        continue  # The current attention module receives its mask in forward().
    compatible_checkpoint[key] = value

gpt_model.load_state_dict(compatible_checkpoint, strict=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
gpt_model.to(device)

GPTModel(
  (tok_emb): Embedding(50257, 768)
  (pos_emb): Embedding(1024, 768)
  (trf_blocks): ModuleList(
    (0-11): 12 x TransformerBlock(
      (norm1): LayerNorm()
      (att): MultiHeadSelfAttention(
        (W_query): Linear(in_features=768, out_features=768, bias=True)
        (W_key): Linear(in_features=768, out_features=768, bias=True)
        (W_value): Linear(in_features=768, out_features=768, bias=True)
        (out_proj): Linear(in_features=768, out_features=768, bias=True)
        (dropout): Dropout(p=0.0, inplace=False)
      )
      (dropout1): Dropout(p=0.0, inplace=False)
      (norm2): LayerNorm()
      (ff): FeedForwardLayer(
        (layers): Sequential(
          (0): Linear(in_features=768, out_features=3072, bias=True)
          (1): GELU(approximate='none')
          (2): Linear(in_features=3072, out_features=768, bias=True)
        )
      )
      (dropout2): Dropout(p=0.0, inplace=False)
    )
  )
  (final_norm): LayerNorm()
  (out_head): Linear(in_features=

In [6]:
from utils import generate_sequence_with_temperature_and_top_k
from transformers import GPT2Tokenizer

# Set the temperature and top-k sampling parameters
temperature = 1.5
top_k = 20

# Define the starting prompt for text generation
start_prompt_text = "Wise men say, only fools rush in"

# Encode using the gpt-2 tokenizer
tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
start_prompt_ids = tokenizer.encode(start_prompt_text, return_tensors="pt").squeeze().tolist()

print("Input IDs:", start_prompt_ids)

# Generate a sequence using the model
output_ids = generate_sequence_with_temperature_and_top_k(gpt_model, start_prompt_ids, max_length=50, device=device, temperature=temperature, top_k=top_k)

# Decode the output IDs back to text
output_text = tokenizer.decode(output_ids)
print("Output Text:", output_text)

/home/savio/Documents/llm-from-scratch/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Input IDs: [54, 786, 1450, 910, 11, 691, 37323, 10484, 287]
Output Text: Wise men say, only fools rush in to do it to you, and then are not even so foolish. The man who is afraid to come back on his knees when he has been called, who wants to live on the banks but wants to make the man come in, is foolish.
